# 🏋️ FitCoach Live Feedback - Google Colab

This notebook runs the FitCoach live feedback system on Google Colab.

**IMPORTANT:**
1. **Runtime**: Select `Runtime -> Change runtime type -> T4 GPU` (recommended) or `A100 GPU` (if available)
2. **Time**: Setup takes ~10-15 minutes (one-time)
3. **Memory**: Requires High-RAM (Pro/Pro+ for best experience)

**GPU Recommendations:**
- **T4 GPU** (16GB VRAM): Works with lightweight mode - **RECOMMENDED for free tier**
- **V100 GPU** (16GB VRAM): Works with lightweight mode
- **A100 GPU** (40GB VRAM): Works with standard mode - **BEST if available**

**Colab Tiers:**
- **Free**: T4 GPU, session limits (~12 hours), may disconnect
- **Pro ($9.99/month)**: Better GPU access, longer sessions
- **Pro+ ($49.99/month)**: A100 access, background execution

---

## Step 1: Check GPU and Setup Environment

Verify you have a GPU and check available memory.

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    
    # Determine which mode to use
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if vram_gb >= 24:
        print("\nRECOMMENDATION: Use STANDARD mode (best quality)")
        recommended_mode = "standard"
    else:
        print("\nRECOMMENDATION: Use LIGHTWEIGHT mode (optimized for your GPU)")
        recommended_mode = "lightweight"
else:
    print("\nERROR: No GPU detected!")
    print("Please go to Runtime -> Change runtime type -> Select T4 GPU")
    recommended_mode = "lightweight"

## Step 2: Clone Repository and Install Dependencies

This will take 2-3 minutes.

In [ ]:
# Clone the repository
!git clone -b kendrick/live-feedback https://github.com/KendrickXie/FitCoach.git
%cd FitCoach

print("\nRepository cloned successfully!")

In [ ]:
# Install dependencies
print("Installing dependencies (this may take 3-5 minutes)...\n")

!pip install -q PyYAML==6.0
!pip install -q datasets==2.14.6
!pip install -q evaluate==0.4.1
!pip install -q opencv-python==4.9.0.80
!pip install -q transformers==4.36.0
!pip install -q accelerate==0.24.1
!pip install -q peft==0.5.0
!pip install -q bitsandbytes==0.43.1
!pip install -q tqdm
!pip install -q rouge_score
!pip install -q bert_score

# Flash attention (optional, may fail on some systems)
try:
    !pip install -q flash-attn==2.5.8 --no-build-isolation
    print("Flash attention installed")
except:
    print("Flash attention failed (optional, will use standard attention)")

print("\nAll dependencies installed!")

## Step 3: Download Model Checkpoints

**This is the longest step (~8-10 minutes)** as we download:
1. LLaMA-2-7B (13GB)
2. 3D CNN weights (500MB)
3. Stream-VLM weights (3.5GB)

**IMPORTANT**: You need a HuggingFace account with access to LLaMA-2.
1. Go to: https://huggingface.co/meta-llama/Llama-2-7b-hf
2. Request access (usually approved instantly)
3. Get your token: https://huggingface.co/settings/tokens

In [ ]:
# Login to HuggingFace
from huggingface_hub import notebook_login

print("Please enter your HuggingFace token:")
print("Get it from: https://huggingface.co/settings/tokens\n")

notebook_login()

In [ ]:
# Download LLaMA-2-7B
print("Downloading LLaMA-2-7B (~13GB, this will take 5-8 minutes)...\n")

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="meta-llama/Llama-2-7b-hf",
    local_dir="./Llama-2-7b-hf",
    local_dir_use_symlinks=False
)

print("\nLLaMA-2-7B downloaded!")

In [ ]:
# Download 3D CNN and Stream-VLM weights
print("Downloading 3D CNN weights...\n")

!mkdir -p ckpts_efficientnet
!wget --quiet --show-progress --no-check-certificate -P ./ckpts_efficientnet \
    https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/efficientnet_3d_cnn_weights.tar.gz
!tar -xzf ./ckpts_efficientnet/efficientnet_3d_cnn_weights.tar.gz -C ./ckpts_efficientnet/

print("\nDownloading Stream-VLM weights (chunked, ~3.5GB)...\n")

!mkdir -p ckpts_streamvlm
!cd ckpts_streamvlm && \
    wget --quiet --show-progress --no-check-certificate \
        https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/streamvlm_weights.tar.gz.aa && \
    wget --quiet --show-progress --no-check-certificate \
        https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/streamvlm_weights.tar.gz.ab && \
    wget --quiet --show-progress --no-check-certificate \
        https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/streamvlm_weights.tar.gz.ac && \
    wget --quiet --show-progress --no-check-certificate \
        https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/streamvlm_weights.tar.gz.ad && \
    wget --quiet --show-progress --no-check-certificate \
        https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/streamvlm_weights.tar.gz.ae && \
    wget --quiet --show-progress --no-check-certificate \
        https://github.com/Qualcomm-AI-research/FitCoach/releases/download/v1.0/streamvlm_weights.tar.gz.af && \
    cat streamvlm_weights.tar.gz.* | tar xzf - && \
    cd ..

print("\nAll model weights downloaded!")

## Step 4: Upload Your Workout Video

You can either:
- **Option A**: Upload a video file from your computer
- **Option B**: Use a sample video URL

**Video requirements:**
- Format: MP4, AVI, MOV
- Length: 30 seconds to 5 minutes (longer videos take more time)
- Content: Person performing exercise (squats, push-ups, etc.)
- Quality: 480p or higher recommended

In [ ]:
# Option A: Upload video from your computer
from google.colab import files

print("Please upload your workout video:")
uploaded = files.upload()

if uploaded:
    video_filename = list(uploaded.keys())[0]
    print(f"\nVideo uploaded: {video_filename}")
else:
    print("\nNo video uploaded. Please run this cell again.")
    video_filename = None

In [ ]:
# Option B: Download a sample video (uncomment to use)
# !wget -O sample_workout.mp4 "YOUR_VIDEO_URL_HERE"
# video_filename = "sample_workout.mp4"
# print(f"Sample video downloaded: {video_filename}")

## Step 5: Run Live Feedback System

Now we'll process your video and generate feedback!

**Select your exercise type** (what exercise is being performed in the video):
- squats
- push-ups
- jumping-jacks
- high-knees
- lunges
- planks
- mountain-climbers
- burpees

**Processing time:** Approximately 1-2x the video length
- 1 minute video -> 1-2 minutes processing
- 3 minute video -> 3-6 minutes processing

In [ ]:
# Set exercise type
exercise_type = "squats"  # Change this to match your video

print(f"Exercise type: {exercise_type}")
print(f"Video file: {video_filename}")
print(f"Mode: {recommended_mode}")

In [ ]:
# Run the live feedback system
if video_filename is None:
    print("ERROR: No video file specified. Please upload a video first.")
else:
    print(f"\nStarting FitCoach Live Feedback System...\n")
    print(f"Processing video: {video_filename}")
    print(f"Exercise: {exercise_type}")
    print(f"Mode: {recommended_mode}\n")
    print("=" * 60)
    
    if recommended_mode == "lightweight":
        config_file = "configs/live_lightweight.yaml"
        script_file = "scripts/live_feedback_lightweight.py"
    else:
        config_file = "configs/live.yaml"
        script_file = "scripts/live_feedback.py"
    
    # Run the system
    !python {script_file} \
        --config {config_file} \
        --video {video_filename} \
        --exercise {exercise_type} \
        --headless
    
    print("\n" + "=" * 60)
    print("\nProcessing complete!")

## Step 6: View Results (Optional)

The feedback is printed above. You can also analyze the output further.

In [ ]:
# Display the video in the notebook
from IPython.display import Video

if video_filename:
    print("Original video:")
    display(Video(video_filename, width=600))

## Step 7: Try Different Videos or Exercises

To test with different videos:
1. Go back to **Step 4** and upload a new video
2. Update `exercise_type` in **Step 5**
3. Re-run **Step 5**

**No need to re-download models!** They're already cached.

## Troubleshooting

### "CUDA out of memory"
**Solutions:**
1. Go to `Runtime -> Disconnect and delete runtime`
2. Restart notebook
3. Make sure you're using lightweight mode (for T4/V100)

### "Model generates empty feedback"
**Solutions:**
1. Check that your video shows the exercise clearly
2. Ensure exercise type matches video content
3. Try a shorter video (30-60 seconds)

### "Session disconnected"
**Solutions:**
1. Free tier has 12-hour session limit
2. Colab may disconnect if idle
3. Consider upgrading to Pro for longer sessions

### "Download failed"
**Solutions:**
1. Check internet connection
2. Re-run the download cell
3. Try again in a few minutes

### "No GPU available"
**Solutions:**
1. Go to `Runtime -> Change runtime type`
2. Select `T4 GPU` (or A100 if available)
3. Click Save
4. Re-run notebook from beginning

## Performance Tips

### Optimize Processing Speed
To make processing faster, you can edit the config file:

In [ ]:
# Optional: Make processing faster (lower quality)
# Uncomment and run this cell to modify config

# import yaml
# 
# config_path = "configs/live_lightweight.yaml"
# with open(config_path, 'r') as f:
#     config = yaml.safe_load(f)
# 
# # Reduce feature extraction rate
# config['evaluator']['sampling_kwargs']['feats_frequency'] = 1  # Was 2
# config['evaluator']['sampling_kwargs']['feedback_interval'] = 15.0  # Was 10
# config['evaluator']['sampling_kwargs']['max_feedback_length'] = 32  # Was 64
# 
# with open(config_path, 'w') as f:
#     yaml.dump(config, f)
# 
# print("Config optimized for speed!")
# print("Note: This will reduce quality slightly but process 2x faster")

## Cleanup (Optional)

Free up disk space after you're done:

In [ ]:
# Clean up downloaded files (saves ~17GB)
# WARNING: You'll need to re-download models if you run this

# !rm -rf Llama-2-7b-hf
# !rm -rf ckpts_efficientnet
# !rm -rf ckpts_streamvlm
# print("Cleanup complete!")